# Week 5: Song Recommendations with Similarity

[Open in Colab](https://colab.research.google.com/github/astarryknight/ai-ml/blob/main/notebooks/week-05-song-recommender.ipynb)

**Estimated time:** 75–90 minutes  
**Finish line:** choose a familiar song and return five recommendations based on understandable audio features.

The model code is short. Most of the work is deciding what “similar” should mean and checking whether the results make sense.

## 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

## 2. Load a small music dataset

The notebook uses the repository copy when available and the raw GitHub file in Colab.

In [ ]:
local_candidates = [
    Path("src/datasets/spotify-2023.csv"),
    Path("../src/datasets/spotify-2023.csv"),
]
local_path = next((path for path in local_candidates if path.exists()), None)
url = "https://raw.githubusercontent.com/astarryknight/ai-ml/main/src/datasets/spotify-2023.csv"
source = local_path if local_path is not None else url

raw = pd.read_csv(source, encoding="latin-1")
print("Rows loaded:", len(raw))

## 3. A feature dictionary

These five numbers form each song's location in a musical feature space. They do not capture everything about music, but people can understand and change them.

In [ ]:
FEATURES = ["bpm", "danceability_%", "valence_%", "energy_%", "acousticness_%"]

feature_dictionary = pd.DataFrame({
    "feature": FEATURES,
    "plain_language_meaning": [
        "tempo in beats per minute",
        "how suitable the track is for dancing",
        "how positive or bright the track sounds",
        "how intense and active the track sounds",
        "how strongly the track uses acoustic sound",
    ],
})
feature_dictionary

## 4. Connect song names to their numbers

Start with recognizable titles. The dictionary below makes the relationship between a song and its feature vector explicit.

In [ ]:
for column in FEATURES:
    raw[column] = pd.to_numeric(raw[column], errors="coerce")

songs = (
    raw[["track_name", "artist(s)_name", *FEATURES]]
    .dropna()
    .drop_duplicates(["track_name", "artist(s)_name"])
    .reset_index(drop=True)
)

sample_titles = ["vampire", "Cruel Summer", "Flowers", "Seven (feat. Latto) (Explicit Ver.)"]
song_dictionary = (
    songs[songs["track_name"].isin(sample_titles)]
    .set_index("track_name")
    [["artist(s)_name", *FEATURES]]
    .to_dict(orient="index")
)
song_dictionary

## 5. Extract the numeric values with NumPy

The dictionary helps us read the data. NumPy gives the model a clean matrix containing one row per song.

In [ ]:
X = songs[FEATURES].to_numpy()

print("Matrix shape:", X.shape)
print("First feature vector:", X[0])

## 6. Scale before measuring distance

Tempo uses values around 60–200 while the percentage features use 0–100. Scaling prevents tempo from dominating only because its numbers use a wider range.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaled_summary = pd.DataFrame({
    "feature": FEATURES,
    "mean_after_scaling": X_scaled.mean(axis=0).round(3),
    "spread_after_scaling": X_scaled.std(axis=0).round(3),
})
scaled_summary

## 7. Train the recommendation model

`NearestNeighbors` stores the song locations and finds the closest rows. Fitting the model takes two lines.

In [ ]:
recommender = NearestNeighbors(n_neighbors=6, metric="euclidean")
recommender.fit(X_scaled)

## 8. Recommend songs by name

In [ ]:
def recommend(title, artist=None, count=5):
    title_match = songs["track_name"].str.casefold() == title.casefold()
    artist_match = True if artist is None else songs["artist(s)_name"].str.contains(artist, case=False, regex=False)
    matches = songs[title_match & artist_match]
    if matches.empty:
        raise ValueError("Song not found. Check the title and artist spelling.")
    seed_index = matches.index[0]
    distances, indices = recommender.kneighbors(X_scaled[seed_index:seed_index + 1], n_neighbors=count + 1)
    result = songs.loc[indices[0][1:], ["track_name", "artist(s)_name", *FEATURES]].copy()
    result.insert(2, "distance", distances[0][1:].round(3))
    return result.reset_index(drop=True)

recommend("vampire", "Olivia Rodrigo")

## 9. Evaluate with human judgment

A smaller distance means the audio features look more similar. It does not guarantee that a listener will enjoy the recommendation.

Try at least three seed songs. For each result, note one recommendation that feels reasonable and one that surprises you. Then identify which feature might explain the match.

In [ ]:
for seed in ["vampire", "Cruel Summer", "Flowers"]:
    print(f"\nRecommendations for {seed}")
    display(recommend(seed))

## 10. Your experiment

Change one choice at a time: remove `bpm`, use only mood features, add `speechiness_%`, or select another seed song. Explain how the recommendations changed.

## Optional depth: musical neighborhoods with K-means

K-means assigns each song to a broad neighborhood. Nearest neighbors still provide the ranked recommendations.

In [ ]:
kmeans = KMeans(n_clusters=8, n_init=10, random_state=42)
songs["cluster"] = kmeans.fit_predict(X_scaled)

seed_title = "vampire"
seed_index = songs.index[songs["track_name"].str.casefold() == seed_title.casefold()][0]
seed_cluster = songs.loc[seed_index, "cluster"]

print("Seed cluster:", seed_cluster)
display(songs[songs["cluster"] == seed_cluster][["track_name", "artist(s)_name"]].head(12))

## 11. Validation

In [ ]:
assert X.shape[1] == len(FEATURES)
assert np.isfinite(X_scaled).all()
assert len(recommend("vampire")) == 5
assert "vampire" not in recommend("vampire")["track_name"].str.casefold().tolist()
print("Validation passed: the recommender returns five ranked songs.")

## Finish line

- A readable song dictionary
- Five ranked recommendations for three seed songs
- A short explanation of one sensible and one surprising result
- One feature change and its effect